# 10 — Industry-Neutral Momentum

Primary test: remove contemporaneous FF49 industry-average momentum from each stock's raw 12–2 momentum, then re-form global quintiles.

Methodology locked before results: FF49 from contemporaneous SICCD, equal-weight industry mean, minimum 10 valid stocks per industry-month, VW Q5−Q1 primary, HAC6/HAC12 inference on Neutral minus Raw spread.


In [ ]:
from pathlib import Path
import sys

ROOT = Path('..').resolve() if Path.cwd().name == 'notebooks' else Path('.').resolve()
sys.path.append(str(ROOT))

from src.clean_monthly import CleaningConfig, build_clean_monthly_panel, build_monthly_history_panel
from src.signals import attach_momentum_to_formation_universe, assign_signal_quintiles
from src.portfolio import monthly_rank_ic, quintile_returns, quintile_return_panel
from src.inference import primary_and_robustness_hac
from src.industry_neutral import attach_ff49, industry_demeaned_momentum, within_industry_percentile_signal, icb_demeaned_momentum, assign_quintiles_from_signal


## 1. Rebuild validated raw momentum dataset


In [ ]:
RAW = ROOT / 'data/raw/monthly_stock_common_equity_2000_2025_prod_v1.csv.gz'
clean, _ = build_clean_monthly_panel(RAW, CleaningConfig())
history = build_monthly_history_panel(RAW)
mom_raw = attach_momentum_to_formation_universe(clean, history)
mom_raw = assign_signal_quintiles(mom_raw)
qret_raw = quintile_returns(mom_raw)
vw_raw = quintile_return_panel(qret_raw, return_col='VW_Return')
ew_raw = quintile_return_panel(qret_raw, return_col='EW_Return')
ic_raw = monthly_rank_ic(mom_raw)


## 2. FF49 mapping coverage sanity check


In [ ]:
mom_ff = attach_ff49(mom_raw)
coverage = mom_ff[['MthCalDt','PERMNO','Momentum_12_2','SICCD','FF49']].copy()
coverage['Mapped'] = coverage['FF49'].notna()
mapping_summary = coverage.groupby('MthCalDt').agg(
    FormationN=('PERMNO','size'),
    ValidMomentum=('Momentum_12_2','count'),
    FF49Mapped=('Mapped','sum')
)
mapping_summary['FF49MappedPct'] = 100 * mapping_summary['FF49Mapped'] / mapping_summary['FormationN']
mapping_summary.tail(12)


## 3. Primary FF49 industry-demeaned momentum


In [ ]:
mom_neutral = industry_demeaned_momentum(mom_ff, min_industry_size=10)
mom_neutral = assign_quintiles_from_signal(mom_neutral, 'Momentum_IndustryNeutral')

eligible = mom_neutral['Momentum_IndustryNeutral'].notna()
print(f'Industry-neutral valid rows: {eligible.sum():,}')
print(f'Coverage vs valid raw momentum: {100 * eligible.sum() / mom_neutral.Momentum_12_2.notna().sum():.2f}%')


## 4. Primary VW spread comparison


In [ ]:
qret_neutral = quintile_returns(mom_neutral)
vw_neutral = quintile_return_panel(qret_neutral, return_col='VW_Return')
ew_neutral = quintile_return_panel(qret_neutral, return_col='EW_Return')
ic_neutral = monthly_rank_ic(mom_neutral, signal_col='Momentum_IndustryNeutral')

cmp = vw_raw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'RawSpread'}).merge(
    vw_neutral[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'NeutralSpread'}),
    on='MthCalDt', how='inner', validate='one_to_one'
)
cmp['Delta_Neutral_Minus_Raw'] = cmp['NeutralSpread'] - cmp['RawSpread']
cmp[['RawSpread','NeutralSpread','Delta_Neutral_Minus_Raw']].mean()


In [ ]:
primary_and_robustness_hac(cmp['Delta_Neutral_Minus_Raw'])


## 5. Secondary: EW and Rank IC


In [ ]:
ew_cmp = ew_raw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Raw'}).merge(
    ew_neutral[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Neutral'}),
    on='MthCalDt', how='inner', validate='one_to_one'
)
ew_cmp['Delta'] = ew_cmp['Neutral'] - ew_cmp['Raw']
print(ew_cmp[['Raw','Neutral','Delta']].mean())
display(primary_and_robustness_hac(ew_cmp['Delta']))

ic_cmp = ic_raw[['MthCalDt','RankIC']].rename(columns={'RankIC':'Raw'}).merge(
    ic_neutral[['MthCalDt','RankIC']].rename(columns={'RankIC':'Neutral'}),
    on='MthCalDt', how='inner', validate='one_to_one'
)
ic_cmp['Delta'] = ic_cmp['Neutral'] - ic_cmp['Raw']
print(ic_cmp[['Raw','Neutral','Delta']].mean())
display(primary_and_robustness_hac(ic_cmp['Delta']))


## 6. Robustness: within-industry percentile ranking


In [ ]:
mom_pct = within_industry_percentile_signal(mom_ff, min_industry_size=10)
mom_pct = assign_quintiles_from_signal(mom_pct, 'Momentum_IndustryPctRank')
vw_pct = quintile_return_panel(quintile_returns(mom_pct), return_col='VW_Return')
pct_cmp = vw_raw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Raw'}).merge(
    vw_pct[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'PctRank'}),
    on='MthCalDt', how='inner', validate='one_to_one'
)
pct_cmp['Delta'] = pct_cmp['PctRank'] - pct_cmp['Raw']
print(pct_cmp[['Raw','PctRank','Delta']].mean())
primary_and_robustness_hac(pct_cmp['Delta'])


## 7. Robustness: CRSP ICBIndustry broad classification


In [ ]:
mom_icb = icb_demeaned_momentum(mom_ff, min_industry_size=10)
mom_icb = assign_quintiles_from_signal(mom_icb, 'Momentum_ICBNeutral')
vw_icb = quintile_return_panel(quintile_returns(mom_icb), return_col='VW_Return')
icb_cmp = vw_raw[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'Raw'}).merge(
    vw_icb[['MthCalDt','Q5_minus_Q1']].rename(columns={'Q5_minus_Q1':'ICBNeutral'}),
    on='MthCalDt', how='inner', validate='one_to_one'
)
icb_cmp['Delta'] = icb_cmp['ICBNeutral'] - icb_cmp['Raw']
print(icb_cmp[['Raw','ICBNeutral','Delta']].mean())
primary_and_robustness_hac(icb_cmp['Delta'])


## Interpretation discipline

The primary estimand is the monthly difference between FF49-neutral and raw VW momentum spreads. A positive delta means industry neutralisation improves the VW spread; a negative delta means raw momentum benefited from industry-level common trends. Secondary specifications must not redefine the primary conclusion.
